In [1]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-text-splitters \
    chromadb \
    sentence-transformers \
    rank_bm25 \
    pypdf \
    flashrank \
    pandas \
    numpy \
    scikit-learn \
    tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-s

In [2]:
import os
import time
import pandas as pd
import numpy as np

# =========================
# PDF LOADER
# =========================
from langchain_community.document_loaders import PyPDFLoader

# =========================
# TEXT SPLITTER
# =========================
from langchain_text_splitters import RecursiveCharacterTextSplitter

# =========================
# EMBEDDINGS
# =========================
from langchain_community.embeddings import HuggingFaceEmbeddings

# =========================
# VECTOR DATABASE
# =========================
from langchain_community.vectorstores import Chroma

# =========================
# RETRIEVERS
# =========================
from langchain_community.retrievers import BM25Retriever

# =========================
# OPENROUTER / OPENAI COMPATIBLE API
# =========================
from langchain_openai import ChatOpenAI

# =========================
# FLASHRANK RERANKER
# =========================
from flashrank import Ranker, RerankRequest

# =========================
# EVALUATION
# =========================
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# PROGRESS BAR
# =========================
from tqdm import tqdm

print("All imports completed successfully.")

/tmp/ipykernel_903/249995393.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All imports completed successfully.


In [3]:
# ==========================================
# OPENROUTER API CONFIGURATION
# ==========================================

from google.colab import userdata

# Get OpenRouter API key securely from Colab Secrets
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY not found. "
        "Please add OPENROUTER_API_KEY in Colab Secrets "
        "and enable notebook access."
    )

# NVIDIA Nemotron 3 Ultra - Free
OPENROUTER_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

print("OpenRouter API key loaded successfully.")
print("Model:", OPENROUTER_MODEL)

OpenRouter API key loaded successfully.
Model: nvidia/nemotron-3-ultra-550b-a55b:free


In [4]:
# ==========================================
# LOAD PDF
# ==========================================

pdf_path = "/content/law pdf.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Total Pages:", len(documents))
print("PDF loaded successfully.")

Total Pages: 402
PDF loaded successfully.


In [5]:
# ==========================================
# RECURSIVE CHARACTER TEXT SPLITTING (CLEAN ENGLISH ARTICLES ONLY)
# ==========================================
import re

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

raw_chunks = text_splitter.split_documents(documents)

chunks = []
for c in raw_chunks:
    text = c.page_content
    # 1. Keep only chunks with substantial English text (excludes Hindi pages)
    if len(re.findall(r'[a-zA-Z]', text)) <= 100:
        continue
    # 2. Exclude Table of Contents / Index chunks (contain multiple dot leaders)
    if len(re.findall(r'\.{4,}|…{2,}', text)) >= 2:
        continue
    # 3. Clean mid-sentence line breaks from PyPDFLoader
    c.page_content = re.sub(r'(?<!\n)\n(?!\n)', ' ', text).strip()
    chunks.append(c)

print("Total raw chunks:", len(raw_chunks))
print("Total clean English Article chunks:", len(chunks))
print("First clean chunk preview:")
print(chunks[0].page_content[:500])

Total raw chunks: 1134
Total clean English Article chunks: 1123
First clean chunk preview:
PREFACE    This is the  sixth pocket size edition of the Constitution of  India in the diglot form. In this edition, the text of the Constitution  of India has been brought up-to-date by incorporating therein all  the amendments up to the Constitution (One Hundred and Sixth  Amendment) Act, 2023. The foot notes below the text indicate the  Constitution Amendment Acts by which such amendments have  been made.  The Constitution (One Hundredth Amendment) Act, 2015  containing details of acquired an


In [6]:
# ==========================================
# HUGGING FACE EMBEDDINGS
# ==========================================

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

/tmp/ipykernel_903/3626235055.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [7]:
# ==========================================
# CREATE CHROMA VECTOR DATABASE
# ==========================================

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="constitution_rag"
)

print("Chroma vector database created successfully.")
print("Total chunks stored:", len(chunks))

Chroma vector database created successfully.
Total chunks stored: 1123


In [8]:
#cell 8
# ==========================================
# DENSE RETRIEVER
# ==========================================

dense_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 10
    }
)

print("Dense retriever created successfully.")
print("Retrieving top 10 chunks for each query.")

Dense retriever created successfully.
Retrieving top 10 chunks for each query.


In [9]:
# ============================================================
# CELL 9 — BM25 RETRIEVER
# ============================================================

bm25_retriever = BM25Retriever.from_documents(chunks)

# Retrieve top 10 documents
bm25_retriever.k = 10

print("BM25 retriever created successfully.")
print("Retrieving top 10 chunks for each query.")

BM25 retriever created successfully.
Retrieving top 10 chunks for each query.


In [10]:
# ============================================================
# CELL 10 — HYBRID RETRIEVER (DENSE + BM25)
# ============================================================

def hybrid_retrieval(query):
    """
    Retrieve documents using both Dense semantic and BM25 keyword search,
    interleaving results so both retrievers contribute candidates to FlashRank.
    """
    dense_docs = dense_retriever.invoke(query)
    bm25_docs = bm25_retriever.invoke(query)

    unique_docs = []
    seen = set()

    # Interleave dense and BM25 documents
    for d_doc, b_doc in zip(dense_docs, bm25_docs):
        for doc in (d_doc, b_doc):
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                unique_docs.append(doc)

    # Return all unique candidates (up to 20) for FlashRank to rerank
    return unique_docs


print("Hybrid retriever updated successfully (Interleaved Dense + BM25).")

Hybrid retriever updated successfully (Interleaved Dense + BM25).


In [11]:
# ============================================================
# CELL 11 — FLASHRANK RERANKER
# ============================================================

# Load the FlashRank reranking model
ranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2",
    cache_dir="/tmp"
)

print("FlashRank reranker loaded successfully.")

ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:02<00:00, 9.01MiB/s]


FlashRank reranker loaded successfully.


In [12]:
# ============================================================
# CELL 12 — FLASHRANK RERANKING FUNCTION
# ============================================================

def rerank_documents(query, documents, top_k=5):
    """
    Rerank retrieved documents using FlashRank.
    """

    # Prepare passages for FlashRank
    passages = []

    for doc in documents:
        passages.append({
            "id": str(len(passages)),
            "text": doc.page_content
        })

    # Create reranking request
    rerank_request = RerankRequest(
        query=query,
        passages=passages
    )

    # Perform reranking
    results = ranker.rerank(rerank_request)

    # Keep top-k results
    results = results[:top_k]

    # Convert FlashRank results back to LangChain documents
    reranked_docs = []

    for result in results:
        text = result["text"]

        for doc in documents:
            if doc.page_content == text:
                reranked_docs.append(doc)
                break

    return reranked_docs


print("FlashRank reranking function created successfully.")

FlashRank reranking function created successfully.


In [13]:
# ============================================================
# CELL 13 — OPENROUTER NEMOTRON LLM
# ============================================================

llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    temperature=0,
    max_tokens=1500,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter LLM initialized successfully.")
print("Model:", OPENROUTER_MODEL)

OpenRouter LLM initialized successfully.
Model: nvidia/nemotron-3-ultra-550b-a55b:free


In [14]:
# ============================================================
# CELL 14 — TEST OPENROUTER NEMOTRON
# ============================================================

test_response = llm.invoke(
    "Explain in one sentence what the Constitution of India is."
)

print("LLM test successful.")
print("\nResponse:")
print(test_response.content)

LLM test successful.

Response:
The Constitution of India is the supreme law of the country, establishing the framework for governance, defining fundamental rights and duties of citizens, and outlining the structure, powers, and functions of the legislature, executive, and judiciary.


In [15]:
# ============================================================
# CELL 15 — COMPLETE RAG ANSWER FUNCTION
# ============================================================
import re

def clean_llm_output(text):
    # Strip <think>...</think> tags if returned by reasoning models
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return text

def answer_question(query, rerank_k=5):
    """
    Complete RAG pipeline:
    Hybrid Retrieval (up to 20) -> FlashRank Reranking (top 5) -> Nemotron
    """

    # Step 1: Hybrid retrieval (returns all unique Dense + BM25 candidates)
    retrieved_docs = hybrid_retrieval(query)

    # Step 2: Rerank retrieved documents down to top 5
    reranked_docs = rerank_documents(
        query,
        retrieved_docs,
        top_k=rerank_k
    )

    # Step 3: Build context
    context = "\n\n".join(
        doc.page_content
        for doc in reranked_docs
    )

    # Step 4: Prompt the LLM (strictly forbid outputting thought process)
    prompt = f"""You are a legal assistant for the Constitution of India.
Provide ONLY the direct, factual answer based strictly on the provided context.
Do NOT include your internal thought process, preamble, or meta-commentary.
If the answer is not in the context, reply: "Answer not found in the provided document."

Context:
{context}

Question:
{query}

Direct Answer:"""

    # Step 5: Generate and clean answer
    response = llm.invoke(prompt)
    final_answer = clean_llm_output(response.content)

    return {
        "question": query,
        "answer": final_answer,
        "retrieved_documents": reranked_docs,
        "context": context
    }


print("Complete RAG answer function updated successfully.")

Complete RAG answer function updated successfully.


In [16]:
# ============================================================
# CELL 16 — TEST COMPLETE RAG PIPELINE
# ============================================================

test_question = "What is the Preamble of the Constitution of India?"

result = answer_question(test_question)

print("=" * 70)
print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nNUMBER OF RERANKED DOCUMENTS:")
print(len(result["retrieved_documents"]))

print("=" * 70)

QUESTION:
What is the Preamble of the Constitution of India?

ANSWER:
WE, THE PEOPLE OF INDIA, having solemnly resolved to constitute India into a SOVEREIGN SOCIALIST SECULAR DEMOCRATIC REPUBLIC and to secure to all its citizens:

JUSTICE, social, economic and political;

LIBERTY of thought, expression, belief, faith and worship;

EQUALITY of status and of opportunity;

and to promote among them all

FRATERNITY assuring the dignity of the individual and the unity and integrity of the Nation;

IN OUR CONSTITUENT ASSEMBLY this twenty-sixth day of November, 1949, do HEREBY ADOPT, ENACT AND GIVE TO OURSELVES THIS CONSTITUTION.

NUMBER OF RERANKED DOCUMENTS:
5


In [17]:
# ============================================================
# CELL 16 — CREATE 20 EVALUATION QUESTIONS
# ============================================================

questions = [
    "What is the Preamble of the Constitution of India?",
    "What does Article 14 of the Constitution provide?",
    "What does Article 15 of the Constitution provide?",
    "What does Article 16 of the Constitution provide?",
    "What does Article 17 of the Constitution provide?",
    "What does Article 18 of the Constitution provide?",
    "What does Article 19 of the Constitution provide?",
    "What does Article 20 of the Constitution provide?",
    "What does Article 21 of the Constitution provide?",
    "What does Article 21A of the Constitution provide?",
    "What does Article 22 of the Constitution provide?",
    "What does Article 23 of the Constitution provide?",
    "What does Article 24 of the Constitution provide?",
    "What does Article 25 of the Constitution provide?",
    "What does Article 26 of the Constitution provide?",
    "What does Article 32 of the Constitution provide?",
    "What does Article 44 of the Constitution provide?",
    "What does Article 51A of the Constitution provide?",
    "What does Article 72 of the Constitution provide?",
    "What does Article 368 of the Constitution provide?"
]

questions_df = pd.DataFrame({
    "question_id": range(1, 21),
    "question": questions
})

questions_df.to_csv(
    "/content/questions_20.csv",
    index=False
)

print("20 questions created successfully.")
print("CSV saved as: /content/questions_20.csv")
print("\nTotal Questions:", len(questions_df))

questions_df

20 questions created successfully.
CSV saved as: /content/questions_20.csv

Total Questions: 20


,question_id,question
0,1,What is the Preamble of the Constitution of In...
1,2,What does Article 14 of the Constitution provide?
2,3,What does Article 15 of the Constitution provide?
3,4,What does Article 16 of the Constitution provide?
4,5,What does Article 17 of the Constitution provide?
5,6,What does Article 18 of the Constitution provide?
6,7,What does Article 19 of the Constitution provide?
7,8,What does Article 20 of the Constitution provide?
8,9,What does Article 21 of the Constitution provide?
9,10,What does Article 21A of the Constitution prov...


In [18]:
# ============================================================
# CELL 17 — RUN ALL 20 QUESTIONS WITH RETRY
# ============================================================

rag_results = []

for _, row in tqdm(
    questions_df.iterrows(),
    total=len(questions_df),
    desc="Processing 20 questions"
):

    question_id = row["question_id"]
    question = row["question"]

    # Retry if OpenRouter/Nemotron is temporarily overloaded
    max_retries = 5
    result = None

    for attempt in range(max_retries):
        try:
            result = answer_question(question)
            break

        except ValueError as e:
            error_message = str(e)

            if "503" in error_message or "overloaded" in error_message.lower():
                wait_time = 10 * (attempt + 1)

                print(
                    f"\nProvider overloaded for Q{question_id}. "
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:
                raise e

    if result is None:
        raise RuntimeError(
            f"Q{question_id} failed after {max_retries} retries."
        )

    rag_results.append({
        "question_id": question_id,
        "question": question,
        "answer": result["answer"],
        "context": result["context"],
        "retrieved_documents": result["retrieved_documents"]
    })


print("\nAll 20 questions processed successfully.")

# Create dataframe
rag_results_df = pd.DataFrame([
    {
        "question_id": r["question_id"],
        "question": r["question"],
        "answer": r["answer"],
        "context": r["context"]
    }
    for r in rag_results
])

print("Total results:", len(rag_results_df))

Processing 20 questions: 100%|██████████| 20/20 [02:33<00:00,  7.69s/it]


All 20 questions processed successfully.
Total results: 20


In [19]:
# ============================================================
# CELL 18 — CREATE REFERENCE ANSWERS WITH RETRY
# ============================================================

# Reset so old truncated reference answers are not reused
reference_answers = []

for i in tqdm(
    range(len(rag_results_df)),
    desc="Creating reference answers"
):
    question = rag_results_df.iloc[i]["question"]
    context = rag_results_df.iloc[i]["context"]

    reference_prompt = f"""Provide ONLY a direct, concise factual answer based strictly on the provided PDF context. Do NOT include any internal thinking process, preamble, or commentary.

PDF Context:
{context}

Question:
{question}

Reference Answer:"""

    response = None

    for attempt in range(5):
        try:
            response = llm.invoke(reference_prompt)
            break
        except ValueError as e:
            error_message = str(e)
            if "503" in error_message or "overloaded" in error_message.lower():
                wait_time = 10 * (attempt + 1)
                print(f"\nProvider overloaded for Q{i+1}. Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                raise e

    if response is None:
        raise RuntimeError(f"Reference answer for Q{i+1} failed after 5 retries.")

    reference_answers.append(clean_llm_output(response.content))

rag_results_df["reference_answer"] = reference_answers

print("\nReference answers created successfully.")
print("Total reference answers:", len(reference_answers))
print("\nExample:")
print("Question:", rag_results_df.iloc[0]["question"])
print("\nReference Answer:")
print(rag_results_df.iloc[0]["reference_answer"])

Creating reference answers: 100%|██████████| 20/20 [01:44<00:00,  5.23s/it]


Reference answers created successfully.
Total reference answers: 20

Example:
Question: What is the Preamble of the Constitution of India?

Reference Answer:
WE, THE PEOPLE OF INDIA, having solemnly resolved to constitute India into a SOVEREIGN SOCIALIST SECULAR DEMOCRATIC REPUBLIC and to secure to all its citizens: JUSTICE, social, economic and political; LIBERTY of thought, expression, belief, faith and worship; EQUALITY of status and of opportunity; and to promote among them all FRATERNITY assuring the dignity of the individual and the unity and integrity of the Nation; IN OUR CONSTITUENT ASSEMBLY this twenty-sixth day of November, 1949, do HEREBY ADOPT, ENACT AND GIVE TO OURSELVES THIS CONSTITUTION.


In [20]:
# ============================================================
# CELL 19 — FIXED EVALUATION (CHUNK-LEVEL NLI + FLEXIBLE MRR)
# ============================================================
import re
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

eval_embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")
nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-small")

def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip()

def calculate_answer_relevance(answer, reference_answer):
    """Semantic alignment between generated answer and ground-truth reference."""
    if not answer.strip() or not reference_answer.strip():
        return 0.0
    embs = eval_embedder.encode([clean_text(answer), clean_text(reference_answer)])
    return float(cosine_similarity([embs[0]], [embs[1]])[0][0])

def calculate_faithfulness(answer, retrieved_documents):
    """Checks NLI entailment of each answer sentence against each individual chunk (avoids 512-token truncation)."""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', clean_text(answer)) if len(s.strip()) > 15]
    chunks_text = [clean_text(doc.page_content)[:1500] for doc in retrieved_documents]

    if not sentences or not chunks_text:
        return 0.0

    sentence_scores = []
    for sent in sentences:
        pairs = [(chunk, sent) for chunk in chunks_text]
        logits = nli_model.predict(pairs)
        probs = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)
        # Take highest entailment probability (index 1) across all 5 retrieved chunks
        best_entailment = float(np.max(probs[:, 1]))
        sentence_scores.append(best_entailment)

    return float(np.mean(sentence_scores))

def calculate_mrr(question, retrieved_documents, reference_answer):
    """Checks rank using flexible Article header regex OR strong semantic match with reference."""
    if not retrieved_documents:
        return 0.0

    match = re.search(r'Article\s+(\d+[A-Z]?)', question, re.IGNORECASE)
    art_num = match.group(1) if match else None

    docs_text = [clean_text(doc.page_content) for doc in retrieved_documents]
    ref_emb = eval_embedder.encode([clean_text(reference_answer)])
    doc_embs = eval_embedder.encode(docs_text)
    similarities = cosine_similarity(ref_emb, doc_embs)[0]

    for rank, (text, sim) in enumerate(zip(docs_text, similarities), start=1):
        # Match Preamble, exact Article header (including amended prefixes like 1[21A.), or high semantic similarity
        if art_num is None and "preamble" in text.lower():
            return 1.0 / rank
        if art_num and re.search(rf'(?:^|\s|\[|\d){re.escape(art_num)}\.\s*[—–\-A-Za-z]', text):
            return 1.0 / rank
        if sim >= 0.65:
            return 1.0 / rank

    return 0.0

evaluation_results = []
for _, row in tqdm(rag_results_df.iterrows(), total=len(rag_results_df), desc="Evaluating"):
    retrieved_docs = rag_results[int(row["question_id"]) - 1]["retrieved_documents"]
    evaluation_results.append({
        "question_id": row["question_id"],
        "question": row["question"],
        "answer_relevance": calculate_answer_relevance(row["answer"], row["reference_answer"]),
        "faithfulness": calculate_faithfulness(row["answer"], retrieved_docs),
        "mrr": calculate_mrr(row["question"], retrieved_docs, row["reference_answer"])
    })

evaluation_df = pd.DataFrame(evaluation_results)
display(evaluation_df[["question_id", "answer_relevance", "faithfulness", "mrr"]])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (16 outputs left out)

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Evaluating: 100%|██████████| 20/20 [00:04<00:00,  4.79it/s]


,question_id,answer_relevance,faithfulness,mrr
0,1,0.988245,0.905466,1.0
1,2,0.639213,0.956069,1.0
2,3,0.602204,0.808681,1.0
3,4,0.625585,0.956069,1.0
4,5,0.645008,0.808680,1.0
5,6,0.650613,0.808680,1.0
6,7,0.551425,0.808680,1.0
7,8,0.632143,0.808680,1.0
8,9,0.647931,0.942086,1.0
9,10,0.592954,0.956069,0.0


In [21]:
# ============================================================
# CELL 20 — FINAL MEAN SCORES AND SAVE CSV
# ============================================================

# Calculate mean values

mean_answer_relevance = evaluation_df["answer_relevance"].mean()
mean_faithfulness = evaluation_df["faithfulness"].mean()
mean_mrr = evaluation_df["mrr"].mean()


# Create summary dataframe

summary_df = pd.DataFrame({
    "Metric": [
        "Mean Answer Relevance",
        "Mean Faithfulness",
        "Mean MRR"
    ],
    "Score": [
        mean_answer_relevance,
        mean_faithfulness,
        mean_mrr
    ]
})


print("========== FINAL EVALUATION RESULTS ==========")

display(summary_df)


# Save complete question-wise results

evaluation_df.to_csv(
    "/content/RAG_20_questions_evaluation_results.csv",
    index=False
)


# Save only summary

summary_df.to_csv(
    "/content/RAG_mean_scores.csv",
    index=False
)


print("\nFiles saved successfully:")
print("1. RAG_20_questions_evaluation_results.csv")
print("2. RAG_mean_scores.csv")

========== FINAL EVALUATION RESULTS ==========


,Metric,Score
0,Mean Answer Relevance,0.654474
1,Mean Faithfulness,0.847033
2,Mean MRR,0.800000



Files saved successfully:
1. RAG_20_questions_evaluation_results.csv
2. RAG_mean_scores.csv
